# Restaurant search — GPU playground (Approaches 6 & 7)

Evaluate and experiment with the GPU-heavy search approaches against the
209-query benchmark (NDCG@5 / Recall@10 / MRR):

- **Approach 6** — cross-encoder scoring every menu item (`crossencoder`)
- **Approach 7** — local LLM as reader via vLLM (`llm_search.py`)
- plus the cheap baselines (`text`, `enhanced`) and optionally ColBERT / SBERT for comparison

**Before running:** *Runtime → Change runtime type → GPU* (L4 or A100 recommended;
a T4 works with a ≤3B model, see the Approach 7 cell).
The repo is private: add a Colab secret named **`GHTOKEN`** (key icon in the left sidebar,
enable *Notebook access*) holding a GitHub fine-grained token with *Contents: read* on
`ahsank/restaurant-search`. Without a token, the notebook uses an existing checkout at
`/content/restaurant-search` if there is one (e.g. an uploaded and unzipped copy).

Run the install cell **first**, before anything imports `torch`.

In [1]:
# --- Environment detection + code checkout -----------------------------------
import os, subprocess, sys

def detect_environment():
    if 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ:
        return 'Colab'
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'Kaggle'
    return 'Local/Other'

ENV = detect_environment()
print(f'Current environment: {ENV}')

REPO = 'ahsank/restaurant-search'
BRANCH = 'main'   # or a feature branch to try unmerged changes

if ENV == 'Colab':
    from google.colab import userdata
    try:
        ghtoken = userdata.get('GHTOKEN')
    except Exception:          # secret not defined / not shared with this notebook
        ghtoken = ''
    root_dir = '/content'
elif ENV == 'Kaggle':
    from kaggle_secrets import UserSecretsClient
    try:
        ghtoken = UserSecretsClient().get_secret('gh-token')
    except Exception:
        ghtoken = ''
    root_dir = '/kaggle/working'
else:
    ghtoken = ''
    root_dir = None            # running from a local checkout: research/colab/ -> repo root

if root_dir is not None and not ghtoken and os.path.isdir(f'{root_dir}/restaurant-search/research'):
    # No token but a checkout is already there (uploaded zip / colab upload): use it as is.
    REPO_DIR = f'{root_dir}/restaurant-search'
    print(f'Using existing checkout at {REPO_DIR}')
elif root_dir is not None:
    REPO_DIR = f'{root_dir}/restaurant-search'
    auth = f'ahsank:{ghtoken}@' if ghtoken else ''
    url = f'https://{auth}github.com/{REPO}.git'
    subprocess.run(['rm', '-rf', REPO_DIR], check=True)
    for branch in dict.fromkeys((BRANCH, 'main')):   # try BRANCH, then main
        r = subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', branch, url, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0:
            print(f'Cloned {REPO}@{branch} -> {REPO_DIR}')
            break
        print(f'Clone of {branch} failed: {r.stderr.replace(ghtoken, "***") if ghtoken else r.stderr}')
    else:
        raise RuntimeError('Could not clone the repo — set the GHTOKEN secret if it is private.')
else:
    REPO_DIR = os.path.abspath('../..')

for p in (f'{REPO_DIR}/research', f'{REPO_DIR}/research/search'):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(REPO_DIR)

Current environment: Colab
Cloned ahsank/restaurant-search@main -> /content/restaurant-search


In [2]:
# --- Install GPU dependencies (run before anything imports torch) -------------
# vLLM pins its own torch; sentence-transformers / pylate then reuse that torch.
# Colab prints dependency-conflict warnings for packages we don't use (RAPIDS, ADK) — harmless.
INSTALL_VLLM = True      # Approach 7
INSTALL_PYLATE = False   # ColBERT (optional comparison)

pkgs = ['sentence-transformers'] + (['vllm'] if INSTALL_VLLM else []) + (['pylate'] if INSTALL_PYLATE else [])
%pip install -q {' '.join(pkgs)}
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72

In [3]:
# --- Shared setup: data, benchmark, engine helper ---------------------------
import json, time
from pathlib import Path
import pandas as pd

import engines
from evaluate import evaluate, load_queries
from lib.data import load_data, judged_restaurant_ids

restaurants, menus = load_data(judged_restaurant_ids())
QUERIES = load_queries()
print(f'{len(restaurants)} restaurants, {sum(map(len, menus.values()))} items, {len(QUERIES)} queries')

RESULTS = {}            # engine label -> evaluate() output
SEARCH_FNS = {}         # engine label -> search_fn(query) for the playground cells

def build_engine(engine, model=None, **flags):
    # engines.build_arg_parser pre-reads sys.argv to pick engine-specific flags;
    # point it at our args instead of the kernel's.
    argv = ['--engine', engine] + (['--model', model] if model else [])
    saved, sys.argv = sys.argv, ['notebook', *argv]
    try:
        args = engines.build_arg_parser('notebook').parse_args(argv)
    finally:
        sys.argv = saved
    for k, v in flags.items():
        setattr(args, k, v)
    return engines.build_search_fn(args, restaurants, menus)

def score(label, search_fn):
    t0 = time.time()
    RESULTS[label] = evaluate(search_fn)
    SEARCH_FNS[label] = search_fn
    o = RESULTS[label]['overall']
    print(f"{label}: NDCG@5={o['ndcg@5']:.4f}  Recall@10={o['recall@10']:.4f}  MRR={o['mrr']:.4f}"
          f"  ({time.time() - t0:.0f}s)")

def summary():
    rows = {k: {m: v['overall'][m] for m in ('ndcg@5', 'recall@10', 'mrr')} for k, v in RESULTS.items()}
    return pd.DataFrame(rows).T.sort_values('ndcg@5', ascending=False).round(4)

def by(key):   # key: 'by_category' or 'by_difficulty'
    return pd.DataFrame({k: {c: m['ndcg@5'] for c, m in v[key].items()} for k, v in RESULTS.items()}).round(3)

8 restaurants, 795 items, 209 queries


## Baselines (CPU, seconds)

In [4]:
score('text', build_engine('text'))
score('enhanced', build_engine('enhanced'))

text: NDCG@5=0.4252  Recall@10=0.4589  MRR=0.4413  (0s)
enhanced: NDCG@5=0.5254  Recall@10=0.5366  MRR=0.5615  (5s)


## Approach 6 — cross-encoder (GPU)

Scores every one of the 795 items per query. ~4 s/query on CPU; a few minutes for the
whole benchmark on a GPU. Swap `CE_MODEL` for any `sentence-transformers` CrossEncoder
checkpoint (e.g. `cross-encoder/ms-marco-MiniLM-L12-v2`, `BAAI/bge-reranker-base`).

In [ ]:
CE_MODEL = 'cross-encoder/ms-marco-MiniLM-L6-v2'
score(f'crossencoder:{CE_MODEL.split("/")[-1]}', build_engine('crossencoder', CE_MODEL))

Optional — ColBERT / SBERT for comparison (set `INSTALL_PYLATE = True` above for ColBERT).

In [ ]:
RUN_EXTRA = False
if RUN_EXTRA:
    score('sbert:all-mpnet-base-v2', build_engine('sbert', 'all-mpnet-base-v2'))
    if INSTALL_PYLATE:
        score('colbert', build_engine('colbert'))

## Approach 7 — local LLM as reader (vLLM, GPU)

One prompt per (query, restaurant): the restaurant's whole menu as a numbered list, the query
last (so vLLM's prefix cache reuses each menu across queries). The model answers with graded
`[item_number, grade]` pairs; restaurants are merged by grade, then rank.

| GPU | Suggested `LLM_MODEL` |
|---|---|
| T4 (16 GB) | `Qwen/Qwen2.5-3B-Instruct`, `microsoft/Phi-3.5-mini-instruct` |
| L4 (24 GB) | `Qwen/Qwen2.5-7B-Instruct` (default) |
| A100 (40/80 GB) | `Qwen/Qwen2.5-14B-Instruct`, `Qwen/Qwen2.5-32B-Instruct-AWQ` |

Gated models (Llama) need an `HF_TOKEN` Colab secret. To try another model, restart the
runtime (vLLM doesn't release GPU memory cleanly) and re-run the setup cells.

In [6]:
import llm_search

LLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
GPU_MEM = 0.85    # leave room for the cross-encoder loaded above

if ENV == 'Colab':
    try:
        os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
    except Exception:
        pass

import torch; torch.cuda.empty_cache()
llm = llm_search.load_llm(LLM_MODEL, gpu_memory_utilization=GPU_MEM)

(MainProcess pid=22162) INFO 10-06 20:00:15 [api_utils.py:286] non-default args: {'max_model_len': 12288, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=22162) INFO 10-06 20:00:15 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=22162) INFO 10-06 20:00:15 [model.py:2109] Using max model len 12288
(MainProcess pid=22162) INFO 10-06 20:00:15 [kernel.py:408] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(EngineCore pid=26759) INFO 10-06 20:00:18 [core.py:129] Initializing a V1 LLM engine (v0.31.0) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=12

Process EngineCore:
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/core.py", line 1437, in run_engine_core
    raise e
  File "/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/core.py", line 1395, in run_engine_core
    engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/vllm/tracing/otel.py", line 175, in sync_wrapper
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/core.py", line 1151, in __init__
    super().__init__(
    ~~~~~~~~~~~~~~~~^
        vllm_config,
        ^^^^^^^^^^^^
    ...<3 lines>...
        internal_dp_balancing,
        ^^^^^^^

(MainProcess pid=22162) INFO 10-06 20:00:19 [utils.py:638] [shutdown] Process manager: send sigterm to process EngineCore


RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [7]:
import sys
import io
from google.colab import userdata
from huggingface_hub import HfApi

# --- 1. Verify Hugging Face Token ---
try:
    hf_token = userdata.get('HF_TOKEN')
    api = HfApi(token=hf_token)
    user_info = api.whoami()
    print(f"✅ HF_TOKEN is valid! Logged in as: {user_info.get('username')}")
except Exception as e:
    print(f"⚠️ HF_TOKEN check failed/not set: {e}")
    print("Note: If loading an open model like Qwen2.5, a valid HF_TOKEN might not be strictly required.")

# --- 2. Monkeypatch fileno() for Colab/Jupyter compatibility with vLLM ---
if not hasattr(sys.stdout, 'fileno'):
    # Fallback to standard descriptor 1 if fileno doesn't exist
    sys.stdout.fileno = lambda: 1

try:
    sys.stdout.fileno()
except (io.UnsupportedOperation, AttributeError):
    # Override the ipykernel OutStream block
    sys.stdout.fileno = lambda: 1

if not hasattr(sys.stderr, 'fileno'):
    sys.stderr.fileno = lambda: 2
try:
    sys.stderr.fileno()
except (io.UnsupportedOperation, AttributeError):
    sys.stderr.fileno = lambda: 2

print("✅ Successfully patched standard stream fileno methods. Re-running vLLM initialization...")

# --- 3. Re-run vLLM Load ---
import torch
torch.cuda.empty_cache()
llm = llm_search.load_llm(LLM_MODEL, gpu_memory_utilization=GPU_MEM)

✅ HF_TOKEN is valid! Logged in as: None
✅ Successfully patched standard stream fileno methods. Re-running vLLM initialization...
(MainProcess pid=22162) INFO 10-06 20:13:48 [api_utils.py:286] non-default args: {'max_model_len': 12288, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=22162) INFO 10-06 20:13:48 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=22162) INFO 10-06 20:13:48 [model.py:2109] Using max model len 12288
(MainProcess pid=22162) INFO 10-06 20:13:48 [kernel.py:408] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(EngineCore pid=30970) INFO 10-06 20:13:52 [core.py:129] Initializing a V1 LLM engine (v0.31.0) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore pid=30970) INFO 10-06 20:14:00 [default_loader.py:484] Loading weights took 4.43 seconds
(EngineCore pid=30970) INFO 10-06 20:14:01 [model_runner.py:407] Model loading took 14.29 GiB memory and 7.413125 seconds
(EngineCore pid=30970) INFO 10-06 20:14:01 [topk_topp_sampler.py:124] Using FlashInfer for top-p & top-k sampling.
(EngineCore pid=30970) INFO 10-06 20:14:01 [torch_utils.py:286] Reducing Torch threads from 6 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
(EngineCore pid=30970) INFO 10-06 20:14:01 [utils.py:343] Using LBNHC KV cache layout.
(EngineCore pid=30970) INFO 10-06 20:14:04 [caching.py:342] reconstructed serializable fn from standalone compile artifacts. num_artifacts=3 num_submods=29
(EngineCore pid=30970) INFO 10-06 20:14:04 [decorators.py:312] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/db37db0d2e6822e1c49f2797109b49fbc4dcf3d5976d939df5cd5a97cd5c723d/rank_0_0/model
(Engine

Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00,  9.53it/s]


(EngineCore pid=30970) INFO 10-06 20:14:15 [model_runner.py:1087] Graph capturing finished in 8 secs, took 0.34 GiB
(EngineCore pid=30970) INFO 10-06 20:14:16 [gpu_worker.py:692] Available KV cache memory: 2.74 GiB
(EngineCore pid=30970) INFO 10-06 20:14:16 [gpu_worker.py:707] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.8500 is equivalent to --gpu-memory-utilization=0.8316 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.8684. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=30970) INFO 10-06 20:14:16 [kv_cache_utils.py:2464] GPU KV cache size: 51,280 tokens, Maximum concurrency for 12,288 tokens per request: 4.17x
(EngineCore pid=30970) INFO 10-06 20:14:16 [kernel_warmup.py:172] JIT kernel warmup starting.
(EngineCore pid=30970) INFO 10-06 20:14:16 [kernel_warmup.py:185] JIT kernel warmup finished in 0.00s.


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:03<00:00, 11.18it/s]


(EngineCore pid=30970) INFO 10-06 20:14:28 [model_runner.py:1087] Graph capturing finished in 11 secs, took 0.24 GiB
(EngineCore pid=30970) INFO 10-06 20:14:28 [gpu_worker.py:892] CUDA graph pool memory: 0.24 GiB (actual), 0.41 GiB (estimated), difference: 0.17 GiB (70.5%).
(EngineCore pid=30970) INFO 10-06 20:14:28 [gpu_worker.py:955] Free memory on device (21.84/22.03 GiB) on startup. Desired GPU memory utilization is (0.85, 18.73 GiB). Actual usage is 14.54 GiB for consumed memory (weights + non-torch), 1.45 GiB for peak activation, and 0.24 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=2527751680` (2.35 GiB) to fit into requested memory, or `--kv-cache-memory=5873200640` (5.47 GiB) to fully utilize gpu memory. Current kv cache memory in use is 2.74 GiB.
(EngineCore pid=30970) INFO 10-06 20:14:28 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=30970) INFO 10-06

In [8]:
# Full benchmark: 209 queries x 8 restaurants = 1,672 prompts (a few minutes on an L4).
# Rankings JSONL can be replayed anywhere: evaluate.py --engine llm --rankings FILE
tag = LLM_MODEL.split('/')[-1].lower()
RANKINGS = Path(f'research/results/llm/{tag}.jsonl')
llm_search.run(QUERIES, restaurants, menus, LLM_MODEL, RANKINGS, llm=llm)
score(f'llm:{tag}', llm_search.replay_search_fn(RANKINGS))

209 queries x 8 restaurants = 1672 prompts


Rendering conversations:   0%|          | 0/1672 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1672/1672 [01:39<00:00, 16.85it/s, est. speed input: 70346.73 toks/s, output: 387.00 toks/s]

Generated in 117.4s (0.56s per query)
Wrote research/results/llm/qwen2.5-7b-instruct.jsonl
llm:qwen2.5-7b-instruct: NDCG@5=0.6157  Recall@10=0.7489  MRR=0.6807  (0s)


## Compare

In [9]:
display(summary())
display(by('by_difficulty'))
by('by_category')

,ndcg@5,recall@10,mrr
llm:qwen2.5-7b-instruct,0.6157,0.7489,0.6807
enhanced,0.5254,0.5366,0.5615
text,0.4252,0.4589,0.4413


,text,enhanced,llm:qwen2.5-7b-instruct
Easy,0.582,0.683,0.706
Medium,0.220,0.345,0.584
Hard,0.131,0.170,0.255


,text,enhanced,llm:qwen2.5-7b-instruct
C01,0.000,0.249,0.662
C02,0.000,0.064,0.552
C03,0.052,0.091,0.295
C04,0.000,0.125,0.469
C05,0.067,0.067,0.720
C06,0.443,0.463,0.672
C07,0.175,0.380,0.544
C08,0.000,0.205,0.323
C09,0.100,0.300,0.544
C10,0.000,0.000,0.197


## Playground — try your own queries

`SEARCH_FNS` holds every engine scored above; the LLM entry replays benchmark rankings,
so the live LLM is wired in separately.

In [10]:
def show(query, k=8):
    fns = dict(SEARCH_FNS)
    if 'llm' in globals():
        fns[f'llm:{LLM_MODEL.split("/")[-1].lower()} (live)'] = lambda q: llm_search.search(q, llm, restaurants, menus)
    cols = {}
    for label, fn in fns.items():
        if label.startswith('llm:') and '(live)' not in label:
            continue
        hits = fn(query)[:k]
        cols[label] = [f"{h['name']} · {restaurants[h['restaurant_id']]['name']}"
                       + (f" [{h['grade']}]" if 'grade' in h else '') for h in hits]
        cols[label] += [''] * (k - len(cols[label]))
    return pd.DataFrame(cols, index=range(1, k + 1))

show('seafood italian')

,text,enhanced,llm:qwen2.5-7b-instruct (live)
1,,Spring Pea Soup · Staple & Fancy,CajunPasta · Bella Italia [2]
2,,Hamachi Crudo · Staple & Fancy,Pappardelle · Staple & Fancy [2]
3,,Prosciutto · Staple & Fancy,Shrimp & Clam Bucatini · Staple & Fancy [2]
4,,Fried Oysters · Staple & Fancy,Spaghetti Carbonara · Bella Italia [1]
5,,Little Gem · Staple & Fancy,Bucatini · Staple & Fancy [1]
6,,Smoked Fish Bruschetta · Staple & Fancy,Mezze Rigatoni · Staple & Fancy [1]
7,,Bucatini · Staple & Fancy,Cacio e Pepe Bucatini · Staple & Fancy [1]
8,,Mezze Rigatoni · Staple & Fancy,


In [ ]:
show('something light and vegetarian for lunch')

## Where does the LLM miss? (per-query NDCG@5, worst first)

In [11]:
llm_key = next((k for k in RESULTS if k.startswith('llm:')), None)
other = next((k for k in RESULTS if k.startswith('crossencoder')), None)
if llm_key:
    pq = pd.DataFrame(RESULTS[llm_key]['per_query']).set_index('query_id')
    if other:
        pq[other] = pd.DataFrame(RESULTS[other]['per_query']).set_index('query_id')['ndcg@5']
    display(pq[pq.n_relevant > 0].sort_values('ndcg@5')
              [['query_text', 'category_name', 'difficulty', 'ndcg@5'] + ([other] if other else [])].head(20))

,query_text,category_name,difficulty,ndcg@5
query_id,,,,
C02-05,smoked meat sandwich,Ingredient generalization,Hard,0.0
C02-04,stone fruit dessert,Ingredient generalization,Hard,0.0
C03-11,charred vegetables,Attribute / implied meaning,Medium,0.0
C04-03,pan seared salmon,Cooking method synonyms,Medium,0.0
C04-02,seared fish,Cooking method synonyms,Medium,0.0
C03-10,tangy dressing salad,Attribute / implied meaning,Medium,0.0
C03-04,light fresh starter,Attribute / implied meaning,Medium,0.0
C03-02,something smoky,Attribute / implied meaning,Hard,0.0
C03-07,umami dish,Attribute / implied meaning,Hard,0.0


In [12]:
# Inspect the raw LLM answer for one query (per restaurant)
QID = 'C03-01'
row = next(json.loads(l) for l in open(RANKINGS) if json.loads(l).get('query_id') == QID)
print(row['query_text'])
for rid, text in row['raw'].items():
    print(f'  {rid:<24} {text[:200]}')

spicy shrimp
  anthonys-homeport        [[2, 2], [32, 2], [51, 1]]
  bella-italia             []
  due-cucina               []
  great-china-restaurant   [[115, 1], [116, 1], [117, 1]]
  juanita-cantina          [[90, 2], [39, 1], [91, 1]]
  mendocino-farms          []
  plaza-garcia             [[87, 1], [107, 2], [118, 2]]
  staple-and-fancy         [[2, 1], [23, 1]]


## Save results

Download `research/results/` (or copy it to Drive) to keep rankings and metrics:
commit the `.jsonl` files so others can replay them with `evaluate.py --engine llm --rankings`.

In [ ]:
summary().to_csv('research/results/summary.csv')
if ENV == 'Colab':
    !cd research && zip -qr /content/results.zip results
    from google.colab import files
    files.download('/content/results.zip')

In [15]:
# --- Score one category, or all of them, for one engine; terminal-style output ---
from collections import defaultdict
from evaluate import evaluate, load_queries, load_relevance, print_report

JUDGMENTS = load_relevance()   # {(restaurant_id, query_id): {item_name: relevance}}

def _search_fn(engine, model, queries, live):
    if engine == 'llm':
        if not live:
            return llm_search.replay_search_fn(RANKINGS)
        if 'llm' not in globals():
            raise RuntimeError('load the model first (the load_llm cell)')
        rows = llm_search.rank_queries(llm, queries, restaurants, menus)   # one batch
        by_text = {r['query_text']: r['results'] for r in rows}
        return lambda q: by_text[q]
    key = f'{engine}:{model}' if model else engine
    if key not in SEARCH_FNS:
        SEARCH_FNS[key] = build_engine(engine, model)
    return SEARCH_FNS[key]

def _avg(rows, key):
    rows = [r for r in rows if r['n_relevant'] > 0]
    return sum(r[key] for r in rows) / len(rows) if rows else 0.0

def _print_query_table(rows):
    w = max(len(r['query_text']) for r in rows)
    print(f"  {'query':<7} {'D':<1} {'NDCG@5':>6} {'R@10':>6} {'MRR':>6} {'#rel':>4}  text")
    print(f"  {'-'*7} {'-'} {'-'*6} {'-'*6} {'-'*6} {'-'*4}  {'-'*w}")
    for r in rows:
        print(f"  {r['query_id']:<7} {r['difficulty'][0]} {r['ndcg@5']:6.3f} {r['recall@10']:6.3f} "
              f"{r['mrr']:6.3f} {r['n_relevant']:4d}  {r['query_text']}")
    print()

def _print_details(rows, search_fn, k):
    # top-k per query: [2]/[1] = judged relevant, [ ] = not; then judged items it missed
    for r in rows:
        if r['n_relevant'] == 0:
            continue
        qid = r['query_id']
        rels = {(rid, name): rel for (rid, q), items in JUDGMENTS.items() if q == qid
                for name, rel in items.items()}
        hits = search_fn(r['query_text'])
        print(f"  {qid}  NDCG@5={r['ndcg@5']:.3f}  «{r['query_text']}»")
        for i, h in enumerate(hits[:k], 1):
            rel = rels.get((h['restaurant_id'], h['name']), 0)
            print(f"     {i}. [{rel or ' '}] {h['name']} · {restaurants[h['restaurant_id']]['name']}")
        shown = {(h['restaurant_id'], h['name']) for h in hits[:k]}
        missed = sorted(((rel, n, rid) for (rid, n), rel in rels.items() if (rid, n) not in shown),
                        reverse=True)
        for rel, n, rid in missed[:5]:
            print(f"        missed [{rel}] {n} · {restaurants[rid]['name']}")
        print()

def score_category(engine, category=None, difficulty=None, model=None,
                   live=True, details=True, k=5, report=None, return_results=False):
    """
    engine: 'text', 'enhanced', 'sbert', 'colbert', 'crossencoder', or 'llm'
    category: e.g. 'C01'; None or 'all' = every category, printed one section per category
    difficulty: 'Easy' / 'Medium' / 'Hard' (None = all)
    model: optional model name for sbert / colbert / crossencoder
    live: for 'llm', True = run the loaded model now; False = replay RANKINGS from the full run
    details: print each query's top-k results and missed relevant items
    report: print evaluate.py's by-category / by-difficulty breakdown at the end
            (default: on for all categories, off for a single one)
    """
    if category == 'all':
        category = None
    if report is None:
        report = category is None

    queries = load_queries(category=category, difficulty=difficulty)
    if not queries:
        raise ValueError(f'no queries for category={category!r} difficulty={difficulty!r}')

    search_fn = _search_fn(engine, model, queries, live)
    res = evaluate(search_fn, category=category, difficulty=difficulty)

    o = res['overall']
    label = engine + (f':{model}' if model else '')
    print('=' * 78)
    print(f"{label}  category={category or 'all'}  difficulty={difficulty or 'all'}  "
          f"({o['n_with_judgments']}/{o['n_queries']} queries judged)")
    print(f"  NDCG@5={o['ndcg@5']:.4f}  Recall@10={o['recall@10']:.4f}  MRR={o['mrr']:.4f}")
    print('=' * 78 + '\n')

    by_cat = defaultdict(list)
    for r in res['per_query']:
        by_cat[r['category_id']].append(r)

    for cat in sorted(by_cat):
        rows = sorted(by_cat[cat], key=lambda r: r['ndcg@5'])   # worst first
        if len(by_cat) > 1:
            print('-' * 78)
            print(f"{cat}  {rows[0]['category_name']}  (n={len(rows)})  "
                  f"NDCG@5={_avg(rows, 'ndcg@5'):.3f}  Recall@10={_avg(rows, 'recall@10'):.3f}  "
                  f"MRR={_avg(rows, 'mrr'):.3f}")
            print('-' * 78)
        _print_query_table(rows)
        if details:
            _print_details(rows, search_fn, k)

    if report:
        print_report(res)
    return res if return_results else None

# Examples:
score_category('llm', 'all', live=False)              # every category, replaying the full-run RANKINGS
# score_category('llm', 'all')                        # every


llm  category=all  difficulty=all  (199/209 queries judged)
  NDCG@5=0.6157  Recall@10=0.7489  MRR=0.6807

------------------------------------------------------------------------------
C01  Cuisine type inference  (n=8)  NDCG@5=0.662  Recall@10=0.832  MRR=0.875
------------------------------------------------------------------------------
  query   D NDCG@5   R@10    MRR #rel  text
  ------- - ------ ------ ------ ----  -----------------------------
  C01-08  H  0.286  0.750  0.500    7  fine dining pasta
  C01-03  M  0.470  0.571  1.000   13  Latin seafood dish
  C01-06  H  0.610  1.000  1.000    7  taqueria style tacos
  C01-02  M  0.682  1.000  0.500    7  Italian pasta with meat sauce
  C01-04  H  0.695  1.000  1.000    4  Tex-Mex combination plate
  C01-07  H  0.817  0.833  1.000   11  rustic Italian comfort food
  C01-01  M  0.869  0.500  1.000   16  Mexican food with shrimp
  C01-05  M  0.869  1.000  1.000    7  farm to table sandwich

  C01-08  NDCG@5=0.286  «fine dining pasta